# 🐘 SQL con PostgreSQL + Python

Notebook de referencia para aprender y repasar SQL desde cero.
Cada sección tiene teoría, ejemplos ejecutables y ejercicios prácticos.

**Motor:** PostgreSQL 16 (Docker)  
**Driver:** psycopg2  
**Schema:** e-commerce (clientes, productos, categorías, pedidos)

---
> ⚠️ **Ejecutar esta primera sección antes de cualquier otra celda**

## 0. Conexión a la base de datos

Antes de ejecutar cualquier query necesitamos conectarnos a PostgreSQL.
Usamos `psycopg2`, el driver más usado en proyectos Python profesionales.

`RealDictCursor` hace que cada fila devuelta sea un diccionario
en lugar de una tupla — más legible y fácil de trabajar.

## 0. Conexión a la base de datos

`psycopg` es el driver oficial de PostgreSQL para Python (v3, la versión moderna).
Antes de ejecutar cualquier query necesitamos importarlo y definir cómo conectarnos.

- `psycopg` — el driver en sí, el que "habla" con PostgreSQL
- `dict_row` — un adaptador que hace que cada fila devuelta sea un diccionario
  (`{"id": 1, "nombre": "Ana"}`) en lugar de una tupla (`(1, "Ana")`)

`CONN_STR` es la cadena de conexión: un string con todos los datos necesarios
para que psycopg encuentre y se autentique con la base de datos.

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `host` | `localhost` | La base corre en esta misma máquina (dentro de Docker) |
| `port` | `5433` | Puerto donde Docker expone PostgreSQL |
| `dbname` | `ecommerce` | Nombre de la base de datos a usar |
| `user` | `alumno` | Usuario de PostgreSQL |
| `password` | `alumno123` | Contraseña del usuario |

In [1]:
# sql-aprendizaje/sql_psycopg.ipynb
import psycopg
from psycopg.rows import dict_row

CONN_STR = "host=localhost port=5433 dbname=ecommerce user=alumno password=alumno123"

## Funciones de utilidad

En lugar de repetir el código de conexión en cada celda, definimos dos funciones
reutilizables que usaremos a lo largo de todo el notebook.

### `obtener_conexion()`

Crea y devuelve una conexión a PostgreSQL con dos opciones importantes:

- `autocommit=True` — cada sentencia SQL se confirma automáticamente en la base
  de datos sin necesidad de llamar a `commit()` manualmente. Ideal para un notebook
  de consultas y aprendizaje; en una aplicación real se manejaría con transacciones explícitas.
- `row_factory=dict_row` — le dice a psycopg que devuelva cada fila como diccionario
  en lugar de tupla, así podemos acceder a los datos por nombre de columna: `fila["nombre"]`.

### `ejecutar_query(sql, params=None)`

Ejecuta cualquier sentencia SQL y devuelve los resultados.

- `conn` — la conexión activa a PostgreSQL. El bloque `with` garantiza que se cierre
  automáticamente al terminar, aunque ocurra un error.
- `cur` (cursor) — el objeto que envía las sentencias SQL al servidor y recibe los resultados.
  Análogo a un puntero dentro de la conexión.
- `sql` — la sentencia SQL a ejecutar, como string.
- `params` — valores a insertar en la query de forma segura (evita SQL injection).
  Se pasa como tupla o diccionario y psycopg los escapa automáticamente.
  Ejemplo: `ejecutar_query("SELECT * FROM clientes WHERE id = %s", (1,))`
- `fetchall()` — recupera todas las filas del resultado como lista. Si la sentencia
  no devuelve filas (un `INSERT` o `CREATE TABLE` por ejemplo), lanza una excepción
  que capturamos devolviendo una lista vacía.

In [2]:
def obtener_conexion():
    """Devuelve una conexion con autocommit y filas como diccionarios."""
    return psycopg.connect(CONN_STR, autocommit=True, row_factory=dict_row)

def ejecutar_query(sql, params=None):
    """Ejecuta una query y devuelve los resultados como lista de diccionarios."""
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            try:
                return cur.fetchall()
            except Exception:
                return []

### Verificación de la conexión

Antes de continuar, comprobamos que todo el stack está funcionando:
Docker corriendo, PostgreSQL levantado, credenciales correctas.

- `with obtener_conexion() as conn` — abre la conexión y la cierra automáticamente
  al salir del bloque, incluso si ocurre un error.
- `with conn.cursor() as cur` — abre un cursor dentro de esa conexión. Un cursor
  es el canal por donde viajan las sentencias SQL hacia el servidor y los resultados
  de vuelta.
- `cur.execute("SELECT version();")` — ejecuta una query nativa de PostgreSQL que
  devuelve la versión del servidor. Es el "hola mundo" de SQL.
- `fetchone()` — recupera solo la primera fila del resultado. A diferencia de
  `fetchall()`, no trae todo sino una sola fila. Como usamos `dict_row`, esa fila
  es un diccionario.
- `version["version"]` — accedemos al valor por nombre de columna. PostgreSQL
  devuelve la columna con el nombre `version`, entonces usamos esa clave.
- El bloque `try/except` captura cualquier error de conexión (Docker apagado,
  credenciales incorrectas, puerto equivocado) y lo muestra en lugar de romper
  el notebook.

Si la celda imprime la versión de PostgreSQL, el entorno está listo.

In [3]:
try:
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT version();")
            version = cur.fetchone()
    print("Conexion exitosa a PostgreSQL")
    print(version["version"])
except Exception as e:
    print(f"Error de conexion: {e}")

Conexion exitosa a PostgreSQL
PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 01. Fundamentos — DDL, DML y tipos de datos

SQL se divide en tres grandes grupos de sentencias según su propósito:

| Grupo | Nombre completo | Para qué sirve | Ejemplos |
|-------|----------------|----------------|---------|
| DDL | Data Definition Language | Definir la estructura de la base | `CREATE`, `ALTER`, `DROP` |
| DML | Data Manipulation Language | Manipular los datos dentro de las tablas | `INSERT`, `SELECT`, `UPDATE`, `DELETE` |
| DCL | Data Control Language | Controlar permisos y accesos | `GRANT`, `REVOKE` |

En esta sección trabajamos DDL: vamos a crear las tablas del schema e-commerce desde cero.

### Tipos de datos en PostgreSQL

Antes de crear una tabla hay que decidir qué tipo de dato va en cada columna.
Los más usados:

| Tipo | Descripción | Ejemplo |
|------|-------------|---------|
| `SERIAL` | Entero autoincremental, típico para IDs | `1, 2, 3, ...` |
| `INTEGER` | Número entero | `42` |
| `NUMERIC(p, s)` | Decimal de precisión fija. `p` = dígitos totales, `s` = decimales | `NUMERIC(10, 2)` → `12345678.90` |
| `VARCHAR(n)` | Texto de longitud variable con límite `n` | `'Ana García'` |
| `TEXT` | Texto sin límite de longitud | descripción larga |
| `BOOLEAN` | Verdadero o falso | `TRUE`, `FALSE` |
| `DATE` | Fecha sin hora | `2024-01-15` |
| `TIMESTAMP` | Fecha con hora | `2024-01-15 10:30:00` |

`SERIAL` es un shorthand de PostgreSQL: crea un entero con una secuencia automática.
En PostgreSQL moderno también existe `GENERATED ALWAYS AS IDENTITY`, pero `SERIAL`
sigue siendo el más común en la práctica.

### Creando el schema e-commerce

Vamos a crear 4 tablas que representan un sistema de comercio electrónico real.
Por ahora sin constraints entre tablas (eso va en la sección 02), solo la estructura básica.

El orden importa: si una tabla referencia a otra, la referenciada debe existir primero.
En este caso creamos `categorias` antes que `productos` porque productos la va a necesitar.

In [4]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS categorias (
                id        SERIAL PRIMARY KEY,
                nombre    VARCHAR(100) NOT NULL,
                activa    BOOLEAN DEFAULT TRUE
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS productos (
                id              SERIAL PRIMARY KEY,
                nombre          VARCHAR(200) NOT NULL,
                precio          NUMERIC(10, 2) NOT NULL,
                stock           INTEGER DEFAULT 0,
                id_categoria    INTEGER,
                creado_en       TIMESTAMP DEFAULT NOW()
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS clientes (
                id          SERIAL PRIMARY KEY,
                nombre      VARCHAR(150) NOT NULL,
                email       VARCHAR(200),
                creado_en   DATE DEFAULT CURRENT_DATE
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS pedidos (
                id          SERIAL PRIMARY KEY,
                id_cliente  INTEGER,
                total       NUMERIC(10, 2),
                estado      VARCHAR(50) DEFAULT 'pendiente',
                creado_en   TIMESTAMP DEFAULT NOW()
            );
        """)

print("Tablas creadas correctamente")

Tablas creadas correctamente


### Verificar las tablas creadas

`information_schema.tables` es una vista del sistema que PostgreSQL mantiene
automáticamente con metadata de todas las tablas existentes.
Filtramos por `table_schema = 'public'` para ver solo las nuestras,
excluyendo las tablas internas del sistema.

In [5]:
tablas = ejecutar_query("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

for tabla in tablas:
    print(tabla["table_name"])

categorias
clientes
pedidos
productos
